# Assignment 7 — NLP I

**Jiya Mahajan | Roll No. 1024170296 | Batch 3Q31 / 3Q3A**


In [ ]:
import nltk
for resource in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4", "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    nltk.download(resource, quiet=True)


In [ ]:
import re, string, math, collections
import pandas as pd
from nltk.tokenize import sent_tokenize, word_tokenize, RegexpTokenizer
from nltk.corpus import stopwords, wordnet
from nltk.stem import PorterStemmer, LancasterStemmer, RegexpStemmer, WordNetLemmatizer
from nltk import pos_tag

raw_tickets = [
    "VPN disconnects every 10 minutes on Windows 11; please check the gateway.",
    "I cannot reset my password after the security update. Contact me at sam@example.com.",
    "Printer queue is stuck in Building 2.1 and shows error code 95.5.",
    "The portal loads slowly at https://help.example.org/status from my laptop.",
    "New employee needs access to the analytics dashboard and shared drive.",
    "Wi-Fi drops during video calls in the north wing; phone 555-013-2098.",
    "Email attachment limit rejects a 24 MB report although the policy says 25 MB.",
    "The software install fails with version 2.0.1 on macOS 14.2.",
    "Please unlock my account; the authenticator code is not arriving.",
    "Projector displays a blue screen when connected to the conference room PC.",
    "Browser bookmarks disappeared after the profile migration yesterday.",
    "Database client reports timeout when I connect to the finance server.",
    "Teams audio is quiet, but the headset works in other applications.",
    "Request permission to install Python for the machine learning course."
]
tickets = raw_tickets
TEST_LINES = ["Version 2.0.1 costs 95.5; email alex@example.com", "Call 555-013-2098 or visit https://example.org/help", "Wi-Fi isn't working in the state-of-the-art lab"]
stop_words = set(stopwords.words("english"))

def process_ticket(text):
    tokens = word_tokenize(text)
    punctuation_tokens = [t for t in tokens if all(c in string.punctuation for c in t)]
    without_punctuation = [t for t in tokens if t not in punctuation_tokens]
    removed_stopwords = [t for t in without_punctuation if t.lower() in stop_words]
    final_tokens = [t.lower() for t in without_punctuation if t.lower() not in stop_words]
    return {"tokens":tokens,"punctuation_tokens":punctuation_tokens,"stopwords":removed_stopwords,"final_tokens":final_tokens}

processed = [process_ticket(ticket) for ticket in tickets]

## Q1

The referenced Colab starter notebook was not supplied. A deterministic local ticket corpus is included; replace `tickets` with the starter notebook data to reproduce its fingerprint-specific results.


In [ ]:
rows=[]
for i,(ticket,result) in enumerate(zip(tickets,processed),1):
    rows.append({"ticket":i,"sentences":len(sent_tokenize(ticket)),"tokens":len(result["tokens"]),"punctuation_tokens":len(result["punctuation_tokens"]),"stopwords":len(result["stopwords"]),"unique_tokens":len(set(result["tokens"])),"final_tokens":len(result["final_tokens"])})
counts = pd.DataFrame(rows)
total = {c:counts[c].sum() for c in counts.columns if c != "ticket"}
display(pd.concat([counts,pd.DataFrame([{"ticket":"TOTAL",**total}])],ignore_index=True))
selected=processed[0]
print("Tokens:",selected["tokens"])
print("Punctuation:",selected["punctuation_tokens"])
print("Stopwords:",selected["stopwords"])
print("Final:",selected["final_tokens"])

## Q2


In [ ]:
raw_text="\n".join(tickets) + "\n" + "\n".join(TEST_LINES)
patterns={"words longer than 5 letters":r"\b[A-Za-z]{6,}\b","numbers including decimals and versions":r"\b\d+(?:\.\d+)+|\b\d+\b","capitalized words":r"\b[A-Z][a-zA-Z]*\b","words beginning with a vowel":r"\b[AEIOUaeiou][A-Za-z]*\b"}
for label,pattern in patterns.items(): print(label, re.findall(pattern,raw_text))
email_re=r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"
url_re=r"https?://[^\s]+"
phone_re=r"(?<!\w)(?:\+?\d{1,2}[-. ]?)?(?:\(?\d{3}\)?[-. ]?)\d{3}[-. ]?\d{4}(?!\w)"
redacted=re.sub(phone_re,"<PHONE>",re.sub(url_re,"<URL>",re.sub(email_re,"<EMAIL>",raw_text)))
print(redacted)

## Q3


In [ ]:
token_pattern=r"https?://[^\s]+|[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}|(?:\+?\d{1,2}[-. ]?)?(?:\(?\d{3}\)?[-. ]?)\d{3}[-. ]?\d{4}|\d+(?:\.\d+)+|\d+(?:\.\d+)?|[A-Za-z]+(?:'[A-Za-z]+)*(?:-[A-Za-z]+(?:'[A-Za-z]+)*)*"
def my_tokenize(text): return re.findall(token_pattern,text)
for line in TEST_LINES: print(line,"\n custom:",my_tokenize(line),"\n nltk:",word_tokenize(line))
for ticket in tickets: print(my_tokenize(ticket))

The custom tokenizer keeps contractions such as `isn't`, hyphenated terms such as `Wi-Fi` and `state-of-the-art`, decimals and versions, emails, URLs, and phone numbers as single tokens. NLTK's general tokenizer commonly splits punctuation and hyphenated forms.

## Q4


In [ ]:
all_final=sorted(set(token for result in processed for token in result["final_tokens"] if token.isalpha()))
porter=PorterStemmer(); lancaster=LancasterStemmer(); regexp=RegexpStemmer(r"(ing|ed|es|s)$", min=4); lemmatizer=WordNetLemmatizer()
def wordnet_pos(tag): return {"J":wordnet.ADJ,"V":wordnet.VERB,"N":wordnet.NOUN,"R":wordnet.ADV}.get(tag[:1],wordnet.NOUN)
tags=dict(pos_tag(all_final))
rows=[]
for word in all_final:
    outputs=[porter.stem(word),lancaster.stem(word),regexp.stem(word),lemmatizer.lemmatize(word,wordnet_pos(tags[word]))]
    if len(set(outputs))>1: rows.append({"word":word,"Porter":outputs[0],"Lancaster":outputs[1],"Regexp":outputs[2],"WordNet lemma":outputs[3]})
display(pd.DataFrame(rows))

The RegexpStemmer removes suffixes found in dataset words such as “working” and “connected.” A suffix-removal rule can over-strip irregular or short words, so only differing forms are displayed.

## Q5


In [ ]:
S1=[t for result in processed for t in result["tokens"]]
S2=[t for t in S1 if not all(ch in string.punctuation for ch in t)]
S3=[t for result in processed for t in result["final_tokens"]]
S4=[porter.stem(t) for t in S3]
def stats(tokens): return {"total tokens":len(tokens),"unique tokens":len(set(tokens)),"TTR":len(set(tokens))/len(tokens) if tokens else 0}
for name, corpus in [("S1",S1),("S2",S2),("S3",S3),("S4",S4)]: print(name,stats(corpus))
print("Top 10 S1:",collections.Counter(S1).most_common(10))
print("Top 10 S3:",collections.Counter(S3).most_common(10))
print("Top 5 bigrams S1:",collections.Counter(zip(S1,S1[1:])).most_common(5))
print("Top 5 bigrams S3:",collections.Counter(zip(S3,S3[1:])).most_common(5))
import matplotlib.pyplot as plt
frequencies=sorted(collections.Counter(S3).values(),reverse=True)
plt.plot([math.log(r) for r in range(1,len(frequencies)+1)],[math.log(f) for f in frequencies],marker=".")
plt.xlabel("log(rank)"); plt.ylabel("log(frequency)"); plt.title("S3 Zipf plot"); plt.grid(True); plt.show()

## Q6
